# Classification II: trees and ensembles

*Decision trees, random forests, cross-validation, and overfitting*

In [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) we built two churn models, logistic regression and KNN, and measured each on 262 test customers that the models had not seen. In this chapter we add two more models: the decision tree and the random forest, an average of many trees. Both are widely used for prediction from tabular data, such as the customer snapshot, in which each row is a customer and each column is a feature. At the end of Chapter 9 we also needed a way to choose a setting such as KNN's k without using the test set. In this chapter we use cross-validation for that purpose. Finally, we compare all four models on accuracy and on the error cost from Chapter 9.

You can run this chapter's notebook in two ways. In the cloud, [**open ch-10-classification-2.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-10-classification-2.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.


In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Decision trees
We start from the Chapter 9 setup: the customer snapshot, with one row per customer and the `churned` label; the same 20 features (input variables); and the same split into a training set and a test set.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR
from sklearn.model_selection import train_test_split

# The Chapter 9 setup: the feature table, the label, and the same 70/30 split.
snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")
X = pd.get_dummies(snap.drop(columns=["customer_id", "pilot", "churned"]),
                   columns=["business_type", "region"])
y = snap["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)

print(X_train.shape, X_test.shape)
print(f"churners: {y_train.sum()} of {len(y_train)} training customers, "
      f"{y_test.sum()} of {len(y_test)} test customers")

As in Chapter 9, 610 customers are used for training and 262 are held out for testing. Of the 610 training customers, 134 churned.

A **decision tree** predicts with a sequence of yes/no questions. In each question, one feature is compared with one number, for example "Has the customer ordered within the last 8.8 weeks?". With such a question, a group of customers is divided into two smaller groups. The question is called a **split**. Each of the two groups can then be split again with a new question. A group that is not split further is a **leaf**. To classify a customer, we answer the questions from the top until the customer reaches a leaf. The leaf's prediction is then the customer's prediction. In terms of the decision boundary of Chapter 9, each question is a straight cut along one feature, so the boundary of a tree is made of cuts parallel to the axes, like a staircase.

The figure compares the two kinds of boundary on the 29 made-up customers of Chapter 9. Both models are fitted to the same customers, and each classifies 26 of the 29 correctly.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier as _Tree

_toy = pd.DataFrame(
    [(1, 20, 0), (2, 14, 0), (3, 22, 0), (2, 8, 0), (4, 17, 0), (5, 25, 0), (6, 19, 0), (3, 4, 0),
     (7, 24, 0), (8, 22, 0), (5, 11, 0), (9, 27, 0), (1, 3, 0), (8, 9, 0), (10, 15, 0),
     (7, 2, 1), (8, 5, 1), (10, 8, 1), (12, 8, 1), (12, 17, 1), (14, 18, 1), (15, 12, 1),
     (13, 4, 1), (16, 22, 1), (18, 16, 1), (17, 7, 1), (9, 2, 1), (6, 8, 1), (11, 12, 0)],
    columns=["weeks", "orders", "churned"])
_X, _y = _toy[["weeks", "orders"]], _toy["churned"]
_models = [("Logistic regression", LogisticRegression(C=1e6, max_iter=5000).fit(_X, _y)),
           ("Decision tree, depth 2", _Tree(max_depth=2, random_state=0).fit(_X, _y))]
_gw, _go = np.linspace(0, 20, 201), np.linspace(0, 30, 151)
_grid = pd.DataFrame([(w, o) for o in _go for w in _gw], columns=["weeks", "orders"])
_fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                     subplot_titles=[name for name, _ in _models])
for _col, (_name, _m) in enumerate(_models, start=1):
    _z = _m.predict(_grid).reshape(len(_go), len(_gw))
    _fig.add_trace(go.Heatmap(x=_gw, y=_go, z=_z, zmin=0, zmax=1, showscale=False, hoverinfo="skip",
                              colorscale=[[0, "#dbe9f6"], [1, "#f6d5cc"]]), row=1, col=_col)
    _fig.add_trace(go.Contour(x=_gw, y=_go, z=_z, contours=dict(start=0.5, end=0.5, coloring="none"),
                              line=dict(color="#333", width=2), showscale=False, showlegend=False, hoverinfo="skip"),
                   row=1, col=_col)
    for _lab, _color, _nm in [(0, "#2b6cb0", "stayed"), (1, "#c0392b", "churned")]:
        _pts = _toy[_toy["churned"] == _lab]
        _fig.add_trace(go.Scatter(x=_pts["weeks"], y=_pts["orders"], mode="markers", name=_nm, legendgroup=_nm,
                                  showlegend=(_col == 1),
                                  marker=dict(size=8, color=_color, line=dict(width=1, color="white"))),
                       row=1, col=_col)
_fig.update_xaxes(title_text="Weeks since last order", range=[0, 20])
_fig.update_yaxes(range=[0, 30])
_fig.update_yaxes(title_text="Orders, last 12 months", row=1, col=1)
_fig.update_layout(height=340, margin=dict(t=50), legend=dict(orientation="h", y=-0.25))
_fig

### How a split is chosen

When a tree is grown, each split is chosen by how mixed the two resulting groups are. A group in which every customer stayed, or every customer churned, is **pure**. A group with equal numbers of churners and stayers is as mixed as a group can be. The usual measure of mixing is the **Gini impurity**. For a group in which a share $p$ of the customers churned,

$$\text{Gini} = 1 - p^2 - (1 - p)^2.$$

In words, the Gini impurity is the chance that two customers drawn at random from the group, with replacement, have different labels. It is 0 for a pure group and 0.5 for an even mix.

At the top of the tree, all 610 training customers form one group. We compute the Gini impurity of that group and of the two groups produced by one candidate split, at 8.8 weeks since the last order:

In [ ]:
def gini(labels):
    """Gini impurity of a group of 0/1 labels."""
    p = labels.mean()                         # share of churners in the group
    return 1 - p**2 - (1 - p)**2

recent = X_train["weeks_since_last_order"] <= 8.8    # True for the left group

rows = []
for name, labels in [("all training customers", y_train),
                     ("left: 8.8 weeks or less", y_train[recent]),
                     ("right: more than 8.8 weeks", y_train[~recent])]:
    rows.append([name, len(labels), labels.sum(), labels.mean(), gini(labels)])

pd.DataFrame(rows, columns=["group", "customers", "churners",
                            "churn rate", "Gini"]).round(3)

In the function, `**` is Python's power operator: `p**2` is $p$ squared. In the loop, we go through three pairs, each made of a name and a column of labels. With `y_train[recent]`, we select the labels of the customers for whom `recent` is True. With `y_train[~recent]`, we select the labels of all other customers.

In the whole training set, 22.0% of the customers churned, and the Gini impurity is 0.343. The left group of 452 customers is nearly pure: 8.0% churned, and its impurity is 0.147. The right group of 158 customers is more mixed, with 62.0% churners and an impurity of 0.471. To score the split as a whole, we average the two impurities, weighting each by the size of its group:

$$\frac{452 \times 0.147 + 158 \times 0.471}{610} = 0.231.$$

With this split, the impurity falls from 0.343 to 0.231. To choose the first split, every feature and every possible split point are tried, and the split with the lowest weighted impurity is kept. On this data, the split at 8.8 weeks since the last order has the lowest weighted impurity of all the candidates. Each of the two groups is then split in the same way, with the search repeated inside the group.

The online edition has an interactive explorer here: a tree grown one split at a time on the 29 made-up customers of Chapter 9, with the tree diagram, the Gini impurity of each group, and the rectangles of the decision boundary after each split. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/tree-grow.html).


The online edition has an interactive explorer here: the 610 training customers plotted along one feature, with a movable split point, the churn rate and Gini impurity of the two resulting groups, and the weighted impurity at every split point for six features. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/tree-split.html).


### The fitted tree

`DecisionTreeClassifier`, from scikit-learn's `tree` module, is a decision tree for classification. With `max_depth=2`, at most two questions are asked on the way to a leaf. With `random_state=0`, ties between equally good splits are broken the same way on every run. `export_text` returns a fitted tree as indented text, with one line per question. With `feature_names=list(X.columns)`, the questions are printed with the column names. With `show_weights=True`, the numbers of training customers who stayed and who churned are also printed for each leaf.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns), show_weights=True))

The first question is the split at 8.8 weeks from the previous section. In each leaf line, the two numbers after `weights:` are the counts of customers who stayed and who churned. The number after `class:` is the leaf's majority class. The following table lists the four leaves:

| Leaf | Customers who reach it | Stayed | Churned | Churn share | Class |
|---|---|---|---|---|---|
| 1 | 2.35 weeks or less since the last order | 237 | 1 | 0.004 | stay |
| 2 | more than 2.35 and at most 8.8 weeks | 179 | 35 | 0.164 | stay |
| 3 | more than 8.8 weeks, average order \$668.26 or less | 36 | 95 | 0.725 | churn |
| 4 | more than 8.8 weeks, average order above \$668.26 | 24 | 3 | 0.111 | stay |

Customers who ordered within 8.8 weeks are split again on recency, at 2.35 weeks. Both resulting leaves have the class "stay". The impurity is lower after this second split. The split is therefore kept, although no class is changed by it. With this split, 238 customers with a single churner are separated from 214 customers of whom 16.4% churned. Customers who had not ordered for more than 8.8 weeks are split on average order value. Those with an average order of \$668.26 or less are predicted to churn. Those with larger orders are predicted to stay.

The figure draws the fitted tree. Each box shows the number of training customers who reach it and how many of them stayed and churned, and each leaf is colored by the class it predicts.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as go

_t = tree.tree_                                   # the depth-2 tree fitted above
_names = list(X.columns)
_pos = {0: (0.5, 3), 1: (0.25, 2), 2: (0.75, 2), 3: (0.125, 1), 4: (0.375, 1), 5: (0.625, 1), 6: (0.875, 1)}
_ids = {}
def _walk(i, slot):                               # map the tree's node ids onto the seven positions
    _ids[slot] = i
    if _t.children_left[i] >= 0:
        _walk(_t.children_left[i], 2 * slot + 1); _walk(_t.children_right[i], 2 * slot + 2)
_walk(0, 0)
_fig = go.Figure()
_w, _h = 0.122, 0.34
_leaf_no = 0
for _slot in range(7):
    _i = _ids[_slot]; _x, _y = _pos[_slot]
    _stay, _churn = (_t.value[_i][0] * _t.n_node_samples[_i]).round().astype(int)
    _n = _stay + _churn
    if _t.children_left[_i] >= 0:                 # a question: lines to its two children, yes on the left
        for _child, _lab in [(2 * _slot + 1, "yes"), (2 * _slot + 2, "no")]:
            _cx, _cy = _pos[_child]
            _fig.add_shape(type="line", x0=_x, y0=_y - _h, x1=_cx, y1=_cy + _h, line=dict(color="#888", width=1.5))
            _fig.add_annotation(x=(_x + _cx) / 2, y=(_y + _cy) / 2, text=_lab, showarrow=False,
                                xshift=-14 if _lab == "yes" else 14, font=dict(size=11, color="#666"))
        _label = f"<b>{_names[_t.feature[_i]]} ≤ {_t.threshold[_i]:.2f}?</b>"
        _fill, _line = "#f6f8fa", "#888"
    else:
        _leaf_no += 1
        _share = _churn / _n
        _label = f"<b>leaf {_leaf_no}: {'churn' if _share > 0.5 else 'stay'}</b><br>churn share {_share:.3f}"
        _fill = f"rgba(192,57,43,{0.25 + 0.6 * _share:.2f})" if _share > 0.5 else "#dbe9f6"
        _line = "#c0392b" if _share > 0.5 else "#2b6cb0"
    _fig.add_shape(type="rect", x0=_x - _w, x1=_x + _w, y0=_y - _h, y1=_y + _h, fillcolor=_fill,
                   line=dict(color=_line, width=1.5))
    _fig.add_annotation(x=_x, y=_y, showarrow=False, font=dict(size=10.5),
                        text=f"{_label}<br>{_n} customers<br>{_stay} stayed, {_churn} churned")
_fig.update_xaxes(visible=False, range=[-0.005, 1.005])
_fig.update_yaxes(visible=False, range=[0.55, 3.45])
_fig.update_layout(height=400, plot_bgcolor="rgba(0,0,0,0)", margin=dict(l=5, r=5, t=5, b=5))
_fig

The tree's predictions are made from its leaves, in two forms. `predict` returns the leaf's majority class, which is the same as applying the default threshold of 0.5 (Chapter 9) to the leaf's churn share. `predict_proba` returns the leaf's share of churners among its training customers, for example 95 / 131 = 0.725 in leaf 3. A depth-2 tree therefore predicts only four different churn probabilities, one per leaf.

No feature scaling is required for trees, unlike KNN in [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html). Each question involves the units of one feature only. The dummy columns are used as they are. With a question such as `region_South Central <= 0.50`, the customers of one region are separated from all others.

In [ ]:
print(f"depth-2 tree: training accuracy {tree.score(X_train, y_train):.3f}, "
      f"test accuracy {tree.score(X_test, y_test):.3f}")

`.score` returns the accuracy of the model's predictions on the features and labels we pass it. The number is the same as the one returned by `accuracy_score` in Chapter 9. The two-question tree is correct for 87.7% of the training customers and for 87.0% of the test customers. On the test set, its accuracy is 2.6 percentage points higher than that of the logistic regression in Chapter 9 (84.4%). On 262 test customers, 2.6 points correspond to about 7 customers. In the model comparison later in this chapter, we check whether the difference is still present when the models are compared by cross-validation, a method that we introduce below.

A fitted tree can also be read as a set of rules. Only leaf 3 predicts churn, so the depth-2 tree reduces to a single rule: if a customer's last order was more than 8.8 weeks ago and the average order is \$668.26 or less, predict churn; otherwise, predict stay. We could apply this rule ourselves, without Python, to any new customer. This is one reason why trees are popular. They are **explainable** by their nature: for each prediction, we can see the questions that led to it. With more complex models, such as the random forest later in this chapter, with its 400 trees, it is often not possible to see why a particular prediction was made.

## Tree depth and overfitting
With `max_depth=2`, the tree has two levels of questions. With a larger depth, the groups are split again and again, until the leaves are nearly pure. We fit trees of increasing depth on the same split and record the number of leaves and the accuracy on both sets:

In [ ]:
results = []
for depth in [1, 2, 3, 4, 6, 8, 10, 12]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0)
    t.fit(X_train, y_train)
    results.append([depth, t.get_n_leaves(),
                    t.score(X_train, y_train), t.score(X_test, y_test)])

depth_table = pd.DataFrame(results, columns=["max_depth", "leaves", "train", "test"]).round(3)
depth_table

`get_n_leaves()` returns the number of leaves of a fitted tree. Training accuracy rises with every added level, from 84.3% at depth 1 to 99.7% at depth 12. Test accuracy is highest at depth 2 (87.0%) and falls to between 79% and 80% from depth 8 on. At depth 12, the tree has 73 leaves for 610 customers, about 8 customers per leaf on average. In a leaf with so few customers, the prediction rests on a handful of labels. Those labels are partly due to chance. As in Chapter 9, a large gap between training and test accuracy is a sign of overfitting (Chapter 9). In the terms of that section, a deep tree has low bias and high variance. A tree of depth 12 is the counterpart of KNN with k = 1 in Chapter 9 (100% training accuracy, 78.6% test accuracy).

In [ ]:
melted = depth_table[["max_depth", "train", "test"]].melt(
    id_vars="max_depth", var_name="set", value_name="accuracy")
px.line(melted, x="max_depth", y="accuracy", color="set", markers=True,
        color_discrete_map={"train": "#0969da", "test": "#cf222e"})

To draw both curves with one call, we first select three columns with two sets of brackets, leaving out `leaves`. Then we reshape the table with `melt`, as in [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html). In the reshaped table, each depth has one row for the training accuracy and one row for the test accuracy.

The online edition has an interactive explorer here: the decision boundary of a tree on two features at every depth from 1 to 12, with its training and test accuracy, showing the boundary break into small boxes around individual customers as the depth increases. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/tree-depth.html).


If we chose the depth from this table, we would repeat the mistake described for k in Chapter 9. The test set would become, in a sense, part of the training process: among eight depths, we would pick the one that happened to suit these 262 customers best, and its test accuracy could no longer be used as an indicator of the model's performance on unseen data. To choose a setting without the test set, we use cross-validation.

## Cross-validation

In **cross-validation** (CV), the training data is divided into equal parts called **folds**. With five folds, the model is fitted five times. Each time, one fold is held out as the **validation fold**, and the model is fitted on the other four. The model is then scored on the validation fold. Every training customer is in exactly one validation fold. Each customer is therefore scored once, by a model that was not fitted on that customer. The CV score is the average of the five fold scores. The test set is not used at any point.

With 610 training customers, each fold has 122 customers. `cross_val_score`, from scikit-learn's `model_selection` module, returns the five fold scores:

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0),
                         X_train, y_train, cv=5)

# --- Look: one accuracy per validation fold ---
print(scores.round(3))
print(f"mean {scores.mean():.3f}, standard deviation {scores.std():.3f}")

`cross_val_score` returns a NumPy array with one accuracy per fold. With `cv=5`, five folds are used. The model is passed unfitted, and a fresh copy of it is fitted in each round. For a classifier, the folds are stratified: each fold has about the same churn rate as the whole training set, as with `stratify=y` in Chapter 9. The following table lists the five rounds:

| Round | Fitted on | Scored on | Accuracy |
|---|---|---|---|
| 1 | folds 2, 3, 4, 5 (488 customers) | fold 1 (122 customers) | 0.861 |
| 2 | folds 1, 3, 4, 5 | fold 2 | 0.852 |
| 3 | folds 1, 2, 4, 5 | fold 3 | 0.910 |
| 4 | folds 1, 2, 3, 5 | fold 4 | 0.877 |
| 5 | folds 1, 2, 3, 4 | fold 5 | 0.852 |
| Mean | | | 0.870 |

The five accuracies range from 0.852 to 0.910. From this spread we can see how much a score on 122 customers depends on which customers are in the fold. We report the standard deviation of the five scores, 0.022, beside the mean.

So far, we have cross-validated a single depth, 2. To choose the depth, we now repeat the cross-validation at each depth. The loop is the same as the one in Chapter 10, with one difference: each depth is scored by its CV accuracy on the training data, and the test set is not used. The training accuracy is printed beside the CV results for comparison.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, 8, 10, 12]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    rows.append([depth, model.score(X_train, y_train),
                 cv_scores.mean(), cv_scores.std()])

pd.DataFrame(rows, columns=["max_depth", "train", "CV mean", "CV std"]).round(3)

The CV mean is highest at depth 2 (0.870) and lower at greater depths, as the test accuracy was. We reach the same depth with the test set unused. Depths 2 and 3 differ by 0.004 in CV mean (0.870 and 0.866). This difference is much smaller than the fold-to-fold standard deviations of about 0.02. A difference this small could easily arise from the particular customers in each fold. When two settings tie in this way, we choose the less flexible one, which has the lower variance, here depth 2.

The online edition has an interactive explorer here: the 610 training customers divided into five folds, with a depth slider, a step-through of the five rounds and their validation accuracies, and the training and CV accuracy at every depth. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/cv-folds.html).


Model settings that we choose before fitting, such as `max_depth` or KNN's k, are called **hyperparameters**. The coefficients of a logistic regression and the split points of a tree are learned from the data during fitting. Hyperparameters, by contrast, are not learned during fitting. We choose hyperparameters by cross-validation on the training data.

With the same loop, we can choose k for KNN, the question left open in Chapter 9:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rows = []
for k in [1, 5, 15, 51, 151]:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    rows.append([k, cv_scores.mean(), cv_scores.std()])

pd.DataFrame(rows, columns=["k", "CV mean", "CV std"]).round(3)

When a pipeline is cross-validated, the scaler is also refitted in each round, on the four fitting folds only. The validation fold is therefore not used in the scaling either. The CV mean is highest at k = 5 (0.831), and k = 15 is close behind (0.821). The difference of 0.010 is smaller than either standard deviation (0.030 and 0.024). With a larger k, each prediction is a vote among more neighbors. The model is therefore less flexible. By the rule above, we keep k = 15 from Chapter 9.

## Random forests

A single tree is unstable, because small changes in the training data can change it a great deal. In the terms of Chapter 9, a single deep tree has high variance. If a few training customers were replaced, a different first split could be chosen, and every question below it could differ. In a **random forest**, many trees are fitted, each in a slightly different way, and their predictions are averaged. The trees are made to differ in two ways, listed in the first two rows of the table:

| Step | What is done | Effect |
|---|---|---|
| Bootstrap sample | Each tree is fitted on 610 customers drawn with replacement from the 610 training customers, as in the bootstrap of [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html). Some customers are drawn more than once, and about a third are not drawn at all. | Each tree is fitted on slightly different data. |
| Random feature subset | At each split, only a random subset of the features is considered. With the default setting, the subset has 4 of the 20 features (the square root of 20, rounded down). | Recency is unavailable at some splits. Other features are used there. |
| Averaging | The forest's churn probability for a customer is the average of the trees' churn probabilities. | Errors that differ from tree to tree partly cancel in the average. |

The online edition has an interactive explorer here: three trees fitted on different bootstrap samples of the 29 made-up customers, and the forest's averaged prediction as trees are added one at a time. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/forest-average.html).


A random forest is fitted with `RandomForestClassifier`, which is imported from scikit-learn's `ensemble` module. An **ensemble** is a model made of several models. With `n_estimators=400`, the forest has 400 trees. With `random_state=0`, the bootstrap samples and the feature subsets are the same on every run.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=400, random_state=0)
forest.fit(X_train, y_train)
print(f"forest: training accuracy {forest.score(X_train, y_train):.3f}, "
      f"test accuracy {forest.score(X_test, y_test):.3f}")

No `max_depth` is set. Each tree is therefore grown until its leaves are pure. A tree grown in this way is correct for every customer in its own bootstrap sample. Each training customer is in the bootstrap samples of about two thirds of the trees. Those trees predict the customer's label correctly. The forest's training accuracy is therefore 100%. From it, we learn nothing about new customers. On the test set, the forest is correct for 83.6% of the customers. In the model comparison below, we measure its accuracy by cross-validation.

To see the averaging, we take one test customer and compute the churn probability from each tree:

In [ ]:
# --- Look: one test customer's churn probability from each of the 400 trees ---
one = X_test.values[11:12]           # the test customer in position 11, as a one-row array
tree_probs = []
for t in forest.estimators_:         # the 400 fitted trees
    tree_probs.append(t.predict_proba(one)[0, 1])

print("first ten trees:", np.array(tree_probs[:10]))
print(f"average over the 400 trees: {np.mean(tree_probs):.4f}")
print(f"forest.predict_proba:       {forest.predict_proba(X_test)[11, 1]:.4f}")
print("actual label:", y_test.values[11])

Line by line, the code above can be read as follows:

- **`one`** is the customer in position 11 of the test set, as a one-row NumPy array. `.values` returns a table's values as an array, and with the slice `[11:12]` we keep one row. The trees were fitted on the feature values without the column names, so we pass the customer in this form.
- **`forest.estimators_`** is the list of the 400 fitted trees, through which the loop runs.
- **`t.predict_proba(one)[0, 1]`** is one tree's churn probability: with `[0, 1]`, we select row 0 and column 1 of the result.
- **`np.array`** returns the first ten probabilities as an array, for compact printing, and **`np.mean`** returns the average of all 400.

Each of the first ten trees predicts a probability of 0 or 1, because the leaves of these trees are pure, and seven of the ten predict churn. Averaged over all 400 trees, the churn probability is 0.5175, the same number as the forest's own probability from `forest.predict_proba`. The forest therefore predicts churn for this customer, although by a narrow margin. According to the actual data, this customer did churn.

A forest has hyperparameters of its own. We set three of them in this chapter:

- **`n_estimators`** is the number of trees. With more trees, the average depends less on the randomness of individual trees, and fitting takes longer.
- **`max_depth`** is the depth limit of each tree, as for a single tree. With the default, `None`, each tree is grown until its leaves are pure.
- **`max_features`** is the number of features considered at each split. With the default, `"sqrt"`, the square root of the number of features is used (4 of 20 here). With a number between 0 and 1, such as 0.5, that share of the features is used (10 of 20).

These hyperparameters are chosen by cross-validation, as the depth of a single tree was.

From a fitted forest, we can also read the **feature importance** of each feature. The Gini impurity is lower after each split, as in the worked example above. For each feature, the decreases from all splits on that feature are added up, with each decrease weighted by the number of customers at the split. The totals are averaged over the 400 trees and scaled to sum to 1. The attribute `feature_importances_` holds one importance per feature, in the order of the columns of `X`.

In [ ]:
importance = pd.Series(forest.feature_importances_, index=X.columns)

# --- Look: the ten largest importances ---
importance.sort_values(ascending=False).head(10).round(3)

In [ ]:
top = importance.nlargest(8).sort_values().reset_index()
top.columns = ["feature", "importance"]
px.bar(top, x="importance", y="feature", orientation="h")

In the chart cell, we sort the eight importances in ascending order with `.sort_values()`, so that the largest bar is drawn at the top. With `reset_index`, the feature names become a column for Plotly.

Recency (`weeks_since_last_order`) accounts for 0.333 of the total, more than three times the share of any other feature. The next seven features, from average order value (0.098) to average discount (0.071), have similar importances. The largest importance among the business-type and region columns is 0.014.

Two cautions apply when we interpret these importances. First, the importances are a property of the model. Recency is used in many splits because it is associated with churn. As in [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), we cannot conclude that recency causes churn from an association alone. Second, a numeric feature can be split at many points, and a True/False column at only one. As a result, numeric features tend to receive higher impurity-based importances. The chart is still useful in a report. From it, the account team can see which features the model uses most. The chart would also show at once if an unwanted feature, such as region, accounted for most of the splitting.

## Four models compared by cross-validation

We now compare the four models of Part IV: logistic regression, KNN with k = 15, the depth-2 tree, and the forest. To keep the test set for the final report, we compare them on the training data by cross-validation. As in Chapter 9, accuracy is only part of the comparison. The business measure is the error cost at the break-even threshold of 0.1 (Chapter 9), with \$60 per offer and \$600 per lost customer.

To compute the error cost by cross-validation, we need a churn probability for every training customer from a model that was not fitted on that customer. `cross_val_predict` returns such probabilities. Inside it, the same five rounds are run as in `cross_val_score`. For each validation fold, the predictions are kept. With `method="predict_proba"`, the predictions are probabilities.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
proba_cv = cross_val_predict(logit, X_train, y_train, cv=5,
                             method="predict_proba")[:, 1]

# --- Look: cross-validated churn probabilities for the first three training customers ---
print(proba_cv[:3].round(3), y_train.values[:3])

`proba_cv` holds 610 churn probabilities, one per training customer. Each comes from the model fitted on the four folds that do not contain that customer. The first training customer stayed and has a churn probability of 0.159. At the break-even threshold of 0.1, this customer would receive an offer. The offer would be a false alarm, a false positive (FP) in the terms of the confusion matrix. The second customer churned and has a probability of 0.898.

We now compute the CV accuracy and the CV error cost for all four models. The two baselines from Chapter 9 are added as the first two rows. The accuracy of the no-offer policy is the majority baseline of the naive model, which predicts "stay" for every customer.

In [ ]:
def error_cost(flagged, labels):
    """Total error cost: $60 per offer, $600 per churner without an offer."""
    tp = (flagged & (labels == 1)).sum()
    fp = (flagged & (labels == 0)).sum()
    fn = ((~flagged) & (labels == 1)).sum()
    return 60 * (tp + fp) + 600 * fn

models = {
    "logistic regression": logit,
    "KNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "tree (depth 2)": DecisionTreeClassifier(max_depth=2, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=400, random_state=0),
}

rows = [["no offers (baseline)", f"{1 - y_train.mean():.3f}", f"${600 * y_train.sum():,}"],
        ["offer to every customer (baseline)", f"{y_train.mean():.3f}", f"${60 * len(y_train):,}"]]
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    proba_cv = cross_val_predict(model, X_train, y_train, cv=5,
                                 method="predict_proba")[:, 1]
    rows.append([name, f"{cv_scores.mean():.3f} ± {cv_scores.std():.3f}",
                 f"${error_cost(proba_cv >= 0.1, y_train):,}"])

pd.DataFrame(rows, columns=["policy", "CV accuracy", "CV error cost at 0.1"])

- **`models.items()`** returns the entries of the dictionary as pairs of key and value. In `for name, model in models.items():`, `name` and `model` hold one model's name and the model object in each pass of the loop.
- **`error_cost(flagged, labels)`** is the cost function from Chapter 9, with the labels as a second argument. With this argument, we can use the same function with the training labels here and with the test labels in the evaluation.

The baselines are computed on the 610 training customers. With no offers, all 134 churners are lost, at a cost of \$80,400. An offer to every customer costs \$36,600. All four models cost less than either baseline.

On cross-validated accuracy, three of the models are effectively tied. Logistic regression (0.869), the tree (0.870), and the forest (0.867) differ by much less than their fold-to-fold standard deviations. The 2.6-point advantage of the tree on the test set is not present in cross-validation. On error cost, the differences are larger. Logistic regression has the lowest cost, \$24,120. The tree costs \$3,300 more, and the forest \$4,320 more. KNN, at \$34,260, is only slightly cheaper than an offer to every customer.

The tree's cost follows from its four leaf probabilities. In the tree fitted on all 610 customers, three of the four leaves have a churn share above 0.1 (0.164, 0.725, and 0.111). At the threshold of 0.1, the tree therefore flags every customer who has not ordered in the last 2.35 weeks. With logistic regression, each customer receives a probability of its own. Among the customers who have not ordered recently, those with low probabilities then receive no offer.

The ranking by accuracy and the ranking by cost differ. As in Chapter 9, we choose on cost, the measure that matches the business. We therefore select logistic regression, at the break-even threshold of 0.1. Only the training data was used for the selection. In the next section, the test set is used once, to report the cost of the selected model on unseen customers.

## Evaluation against two baselines

|  |  |
|---|---|
| **Metric** | total error cost on the test set at the break-even threshold of 0.1, with \$60 per offer and \$600 per lost customer, assuming that a churner who receives an offer stays, with test accuracy reported alongside. |
| **Test set** | the 262 held-out customers, used once, after the model was selected by cross-validation. |
| **Baselines** | no offers, so every churner is lost; and an offer to every customer, so no churner is lost. |



In [ ]:
rows = [["no offers (baseline)", 1 - y_test.mean(), f"${600 * y_test.sum():,}"],
        ["offer to every customer (baseline)", y_test.mean(), f"${60 * len(y_test):,}"]]
for name, model in models.items():
    model.fit(X_train, y_train)                    # fitted on all 610 training customers
    proba = model.predict_proba(X_test)[:, 1]
    rows.append([name, model.score(X_test, y_test),
                 f"${error_cost(proba >= 0.1, y_test):,}"])

pd.DataFrame(rows, columns=["policy", "test accuracy", "test error cost at 0.1"]).round(3)

The selected model, logistic regression, costs \$14,040 on the test set, the same figure as in Chapter 9. This is about 60% below the cost of sending no offers (\$34,800) and about 11% below the cost of an offer to every customer (\$15,720).

On the test set, the tree has both the highest accuracy (0.870) and the lowest cost (\$12,540). In cross-validation, the tree cost more than logistic regression. Each ranking rests on a limited number of customers: 610 in cross-validation and 262 in the test set. On 262 customers, a difference of \$1,500 corresponds to 25 offers, or to 2.5 lost customers. A difference of this size could arise from the particular customers in the test set. We keep the selection made by cross-validation. If we switched to the tree now, we would be selecting the model on the test set. The tree's test cost of \$12,540 would then no longer be a measurement on unseen data.

KNN costs \$15,720 on the test set, the same as an offer to every customer. On this test set, KNN has no advantage over the policy that requires no model. As in Chapter 9, all of these costs are computed on the assumption that a churner who receives an offer stays. This assumption can be tested only with an experiment, as described at the end of Chapter 9.

## The decision this informs

The account team keeps the weekly list from Chapter 9: every customer whose churn probability from the logistic model is above the break-even threshold of 0.1. On this data, the cross-validated error cost of the tree and of the forest was higher than that of logistic regression. This data has one dominant feature, recency, and only 610 training customers. For such data, a more flexible model may add little. The depth-2 tree is still useful for explaining the list. From its questions, the main pattern can be stated in one sentence: customers who have not ordered for more than about two months, and whose average order is under about \$670, are the most likely to churn. This sentence can be given to the account team with the list. When a new snapshot is available, for example a year later, we repeat the cross-validation comparison on the new snapshot before producing the next list of customers.

## Exercises


### Build lab

Tune the random forest by cross-validation. On the training set only, compute the five-fold CV accuracy of `RandomForestClassifier(n_estimators=400, random_state=0)` for each combination of `max_depth` (3, 6, and `None`) and `max_features` (`"sqrt"` and 0.5), six combinations in all. Report the CV mean and standard deviation of each. Choose the combination with the highest CV mean accuracy, unless the second-highest is within one standard deviation of it; in that case, treat the two as tied and choose the less flexible one, as we did for the tree depth. Then fit the chosen forest on the training set and report its test accuracy once.

### Evaluate lab

Compare your tuned forest with the chapter's models on the business measure. First, compute its CV error cost at the break-even threshold of 0.1 with `cross_val_predict`, and compare it with the chapter's CV error costs (logistic regression \$24,120, default forest \$28,440). Then compute its test error cost at thresholds 0.5 and 0.1, and compare it with the two baselines (\$34,800 and \$15,720). Use this rule for the recommendation: replace the logistic model only if the tuned forest has a lower CV error cost at 0.1 than logistic regression. State in two sentences whether the setting chosen for accuracy also lowered the CV error cost, and whether the logistic model is replaced under the rule.

A starter notebook for this lab is provided. It restates the task, reproduces the objects from this chapter that the lab builds on, and marks the cells you complete. [**Open ch-10-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-10-lab-starter.ipynb), or download it from the course page in Blackboard. Colab opens a notebook from GitHub read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.
